# Test on ERA5

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt
import os

import rasterio as rio
from pyproj import CRS

In [ ]:
from etdataset import era5
from etdataset.dem import get_dem_from_roi
from etdataset.interpolation import create_grid_dataset
from etdataset.plot import plot_dataset, plot_dem
from etdataset.utils import get_utm_bbox_from_roi, read_product_list

In [ ]:
import logging

from etdataset.logging import LoggerManager

LoggerManager.set_level(logging.DEBUG)

## Inputs

In [ ]:
product_list = read_product_list(
    "landsat_products.csv", CRS(4326), "overlap_geometry"
)

In [ ]:
product_list

In [ ]:
output = "out"
os.makedirs(output, exist_ok=True)

## Download required data

In [ ]:
bounds = rio.coords.BoundingBox(*product_list.loc[0].geometry.bounds)
crs = CRS(4326)
bounds

In [ ]:
date = dt.datetime(2024, 10, 21, 10, 33)

Download ERA5 dataset 

In [ ]:
era5.download(date=date, dataset=era5.ERA5Dataset.ERA5, path=output)

In [ ]:
era5.download(date=date, dataset=era5.ERA5Dataset.ERA5LAND, path=output)

## Read ERA5 data

In [ ]:
raw_data = era5.read(
    product=os.path.join(output, "ERA5_data", "download_era5_2024-10-21.zip")
)

In [ ]:
raw_data

In [ ]:
diff = (
    raw_data[era5.ERA5Var.SURFACE_SOLAR_RADIATION_DOWNWARD_CLEAR_SKY.key]
    - raw_data[era5.ERA5Var.SURFACE_SOLAR_RADIATION_DOWNWARD.key]
)
diff.plot()

## Add ERA5 data

Artificial data

In [ ]:
# Create a test dataset
grid = create_grid_dataset(bounds, crs, 0.1)
grid["height"] = grid["grid"].copy()
# Add attributes
grid.attrs["vis_date"] = date.date()
grid.attrs["vis_time"] = date.time()
grid.attrs["tir_date"] = date.date()
grid.attrs["tir_time"] = date.time()
grid

In [ ]:
data_era5 = era5.add(data=grid, dataset=era5.ERA5Dataset.ERA5, path=output)

In [ ]:
data_era5.data_vars

In [ ]:
data_era5.rsd_era5.plot()

In [ ]:
data_era5.ta.plot()

In [ ]:
data_era5land = era5.add(
    data=grid, dataset=era5.ERA5Dataset.ERA5LAND, path=output
)

In [ ]:
data_era5land.data_vars

In [ ]:
data_era5land.rsd_era5land.plot()

In [ ]:
data_era5land.ta.plot()

In [ ]:
data_mixte = era5.add(
    data=grid,
    dataset=era5.ERA5Dataset.ERA5LAND,
    variables=[era5.ERA5Var.TEMPERATURE, era5.ERA5Var.DEWPOINT_TEMPERATURE],
    path=output,
)
data_mixte = era5.add(
    data=data_mixte,
    dataset=era5.ERA5Dataset.ERA5,
    variables=[
        era5.ERA5Var.SURFACE_SOLAR_RADIATION_DOWNWARD_CLEAR_SKY,
        era5.ERA5Var.SURFACE_THERMAL_RADIATION_DOWNWARD_CLEAR_SKY,
    ],
    path=output,
)

In [ ]:
data_mixte.data_vars

In [ ]:
# with error
data_error = era5.add(
    data=grid,
    dataset=era5.ERA5Dataset.ERA5LAND,
    variables=[
        era5.ERA5Var.SURFACE_SOLAR_RADIATION_DOWNWARD_CLEAR_SKY,
        era5.ERA5Var.SURFACE_THERMAL_RADIATION_DOWNWARD_CLEAR_SKY,
    ],
    path=output,
)
data_error.data_vars

In [ ]:
# with error
data_rsd = era5.add(
    data=grid,
    dataset=era5.ERA5Dataset.ERA5,
    variables=[
        era5.ERA5Var.SURFACE_SOLAR_RADIATION_DOWNWARD,
    ],
    path=output,
)
data_rsd_cs = era5.add(
    data=grid,
    dataset=era5.ERA5Dataset.ERA5,
    variables=[
        era5.ERA5Var.SURFACE_SOLAR_RADIATION_DOWNWARD_CLEAR_SKY,
    ],
    path=output,
)
diff = data_rsd_cs["rsd_era5"] - data_rsd["rsd_era5"]
diff.plot()

# Get DEM

In [ ]:
# ROI
roi_path = os.path.join("data", "Zone_Senegal_Centre.shp")
roi_bbox, roi_crs = get_utm_bbox_from_roi(roi_path)

In [ ]:
mnt_path = os.path.join(os.environ["MNT_PATH"], "DEM_Copercinus_30m/")
data = get_dem_from_roi(
    roi_bbox=roi_bbox, roi_crs=roi_crs, base_dir=mnt_path, resolution=100
)

In [ ]:
# Add attributes
data.attrs["vis_date"] = date.date()
data.attrs["vis_time"] = date.time()
data.attrs["tir_date"] = date.date()
data.attrs["tir_time"] = date.time()
data

In [ ]:
plot_dem(data)

In [ ]:
updated_data = era5.add(
    data=data,
    dataset=era5.ERA5Dataset.ERA5LAND,
    variables=[era5.ERA5Var.TEMPERATURE, era5.ERA5Var.DEWPOINT_TEMPERATURE],
    path=output,
)
updated_data = era5.add(
    data=updated_data,
    dataset=era5.ERA5Dataset.ERA5,
    variables=[
        era5.ERA5Var.SURFACE_SOLAR_RADIATION_DOWNWARD_CLEAR_SKY,
        era5.ERA5Var.SURFACE_THERMAL_RADIATION_DOWNWARD_CLEAR_SKY,
    ],
    path=output,
)

In [ ]:
plot_dataset(updated_data)